# Запуск расчёта метрик MT-модели (osiris)

Запускает `mt_ddp_metrics.py`: все выходы модели (`cr_general`, `cr_adj`, `ctr`, при наличии `cr`) за один проход, метрики по каналам и сравнение `cr_adj` с `cr_general` (бутстрэп). Результат — Excel в каталоге метрик и таблицы в ноутбуке.

Порядок: 1) импорты → 2) Kerberos → 3) параметры → 4) preflight → 5) задача → 6) запуск → 7) мониторинг и результаты. Режим `check` считает на малой доле файлов (быстрая проверка), `full` на всех.

In [1]:
import os
import sys
import re
import glob
import json
import time
import shutil
import getpass
import subprocess

import pandas as pd

sys.path.insert(0, os.path.abspath("/opt/clients"))
import osiris

## 2. Kerberos / Hadoop (один раз за сессию)

In [2]:
NFS = "/home/datalab/nfs"
PREPARE_ENV = True   # False, если ticket и конфиги уже подготовлены

if PREPARE_ENV:
    for name in ("hadoop", "nss", "krb5cc", "krb5.conf"):
        path = f"{NFS}/{name}"
        if os.path.isdir(path):
            shutil.rmtree(path)
        elif os.path.exists(path):
            os.remove(path)
    shutil.copytree("/etc/hadoop/conf", f"{NFS}/hadoop", dirs_exist_ok=True)
    shutil.copy("/etc/krb5.conf", f"{NFS}/krb5.conf")
    subprocess.run(["kinit"], input=getpass.getpass("Пароль для kinit: ") + "\n", text=True, check=True)
    ticket = max(glob.glob("/tmp/krb5cc_*"), key=os.path.getmtime)
    shutil.copy(ticket, f"{NFS}/krb5cc")
    os.makedirs(f"{NFS}/nss", exist_ok=True)
    shutil.copy("/etc/passwd", f"{NFS}/nss/")
    print("Kerberos/Hadoop подготовлены, ticket:", ticket)

Пароль для kinit:  ········


Password for 21835579_omega-sbrf-ru@DF.SBRF.RU: 
Kerberos/Hadoop подготовлены, ticket: /tmp/krb5cc_1000290000


## 3. Параметры

In [13]:
# ============ РЕЖИМ ============
RUN_MODE = "full"        # "check" — малая доля файлов; "full" — вся выборка
DRY_RUN = False            # True: только печать аргументов, задача не создаётся

PRESETS = {
    "check": dict(file_fraction=0.02, max_batches=0, num_nodes=1, num_gpus=4, suffix="_check"),
    "full":  dict(file_fraction=1.0,  max_batches=0, num_nodes=3, num_gpus=4, suffix=""),
}

# ============ ПУТИ ============
# TRAIN_SCRIPT_PATH = "/home/datalab/nfs/Omni/TRAIN OMNI/5 MDMH/CR CTR EXPERTS CHNL score/mdmh_cr_ctr_base_trai.py"
METRICS_SCRIPT_PATH = "/home/datalab/nfs/Omni/SCORE OMNI/MDMH score/CR CTR EXPERTS CHNL score/mt_ddp_metrics.py"   # положите скрипт рядом с обучающим или укажите путь
LOCAL_ROOT = "/home/datalab/nfs/Omni/"
CONFIG_PATH = "/home/datalab/nfs/Omni/configs_weights/omni_config.json"
DATASET_PATH = ""
WEIGHTS_DIR = f"{LOCAL_ROOT.rstrip('/')}/omnichannel_artifacts/omni/weights"
METRICS_DIR = f"{LOCAL_ROOT.rstrip('/')}/omnichannel_artifacts/omni/metrics"

# ============ МОДЕЛЬ (из обучения) ============
MDL_NAME = "omni_val_cr_ctr_mt_adapter"            # имя обученной модели, включая суффикс режима
LAST_EPOCH = 1                                           # индекс последней эпохи (EPOCHS - 1)
CKPT_NAME = f"{MDL_NAME}_ep{LAST_EPOCH}_weights.pt"      # лучше *_weights.pt
MT_CONFIG_NAME = f"{MDL_NAME}_mt_config.json"            # сохраняется скриптом обучения рядом с весами

# ============ ДАННЫЕ ДЛЯ ОЦЕНКИ ============
TRAIN_TABLE = "sdi_cr_ctr_dataset_val_fin"               # только для предупреждения об оценке на обучающей выборке
VAL_TABLE = ""                                           # ОБЯЗАТЕЛЬНО: таблица для оценки
TST_TABLE = ""
OOT_TABLE = "sdi_cr_ctr_dataset_oot"
EVAL_SPLITS = "oot"                                      # val,tst,oot через запятую
DATASET_SIZE = 202_666_520                               # число строк сплита (только для прогресс-бара)
DATA_FORMAT = "raw"
TARGET_COLUMNS = "trgt_fin,trgt_ctr"                     # в том же порядке, что при обучении

# ============ МЕТРИКИ ============
HEADS = "auto"                                           # auto или список: cr_general,cr_adj,ctr
MODES = "general,product,channel,product_channel"
N_BOOT = 200
BOOT_MAX_ROWS = 300000
NO_COMPARE = False
RESULT_NAME = "metrics"

# ============ ПРОЧЕЕ ============
BATCH_SIZE = 128
SEED = 42
NUM_WORKERS = 2
USE_EMB_FLAGS = """{"romashka_emb": true,
        "evk_emb": true,
        "prod_hier_emb": true,
        "prod_desc_emb": true
    }"""
EVK_WEIGHTS_PATH = "evk_weights.pt"
DLT_WEIGHTS_PATH = "dlt_weights.pt"
POOL = "b2c"

# ============ ПРОИЗВОДНЫЕ ============
assert RUN_MODE in PRESETS, f"RUN_MODE должен быть одним из {list(PRESETS)}"
P = PRESETS[RUN_MODE]
FILE_FRACTION, MAX_BATCHES = P["file_fraction"], P["max_batches"]
NUM_NODES, NUM_GPUS = P["num_nodes"], P["num_gpus"]
NUM_PROCESS = NUM_NODES * NUM_GPUS
RESULT_TAG = RESULT_NAME + P["suffix"]
print(f"mode={RUN_MODE} model={MDL_NAME} ckpt={CKPT_NAME} splits={EVAL_SPLITS} procs={NUM_PROCESS} file_fraction={FILE_FRACTION}")

mode=full model=omni_val_cr_ctr_mt_adapter ckpt=omni_val_cr_ctr_mt_adapter_ep1_weights.pt splits=oot procs=12 file_fraction=1.0


## 4. Preflight

Проверяет наличие скрипта метрик, чекпоинта и `mt_config.json`, разбирает конфиг и показывает, какие выходы будут оценены, проверяет число колонок таргета и таблицы для сплитов.

In [14]:
def _fail(msg):
    raise AssertionError("PREFLIGHT: " + msg)

def preflight():
    for name in ("USE_EMB_FLAGS",):
        try:
            json.loads(globals()[name])
        except Exception as e:
            _fail(f"{name} не парсится как JSON: {e}")
    if not os.path.exists(METRICS_SCRIPT_PATH):
        _fail(f"нет METRICS_SCRIPT_PATH={METRICS_SCRIPT_PATH}")
    text = open(METRICS_SCRIPT_PATH, encoding="utf-8").read()
    for tok in ("--mt-config", "--heads", "collect_predictions", "compare_table"):
        if tok not in text:
            _fail(f"в {METRICS_SCRIPT_PATH} нет '{tok}': это не тот mt_ddp_metrics.py")
    ckpt = f"{WEIGHTS_DIR}/{CKPT_NAME}"
    if not os.path.exists(ckpt):
        cands = sorted(glob.glob(f"{WEIGHTS_DIR}/{MDL_NAME}_ep*_weights.pt"))
        _fail(f"нет чекпоинта {ckpt}. Найдено для модели: {[os.path.basename(c) for c in cands]}")
    mt_path = f"{WEIGHTS_DIR}/{MT_CONFIG_NAME}"
    if not os.path.exists(mt_path):
        _fail(f"нет {mt_path}: файл сохраняет скрипт обучения рядом с весами")
    cfg = json.load(open(mt_path))
    cps = cfg["channels_per_task"]
    no_towers = cfg.get("tower_type", "per_channel") == "none"
    heads = {t: t for t in cfg["tasks"] if not (no_towers and cps[t] > 1)}
    heads.update({f"{t}_general": t for t in cfg.get("general_tasks", [])})
    heads.update({f"{t}_adj": t for t in cfg.get("adapter_tasks", [])})
    if HEADS != "auto":
        sel = [h.strip() for h in HEADS.split(",") if h.strip()]
        unknown = [h for h in sel if h not in heads]
        if unknown:
            _fail(f"HEADS: неизвестные выходы {unknown}; доступны {list(heads)}")
        heads = {h: heads[h] for h in sel}
    cols = [c.strip() for c in TARGET_COLUMNS.split(",")]
    need = max(cfg["target_cols"][t] for t in heads.values()) + 1
    if len(cols) < need:
        _fail(f"TARGET_COLUMNS={cols} короче, чем нужно ({need} колонок) для выходов {list(heads)}")
    tables = {"val": VAL_TABLE, "tst": TST_TABLE, "oot": OOT_TABLE}
    for s in [x.strip() for x in EVAL_SPLITS.split(",") if x.strip()]:
        if s not in tables:
            _fail(f"неизвестный сплит {s}; доступны {list(tables)}")
        if not tables[s]:
            _fail(f"для сплита '{s}' не задана таблица ({s.upper()}_TABLE)")
        if tables[s] == TRAIN_TABLE:
            print(f"WARN: сплит '{s}' = обучающая таблица {TRAIN_TABLE}: метрики будут оптимистичны")
    pairs = [f"{t}_adj vs {t}_general" for t in cfg.get("adapter_tasks", []) if f"{t}_adj" in heads and f"{t}_general" in heads]
    print("выходы для оценки:", heads)
    print("сравнение:", pairs if (pairs and not NO_COMPARE) else "не будет (нет пары cr_adj/cr_general или NO_COMPARE)")
    print("PREFLIGHT: OK")
    return heads

HEADS_RESOLVED = preflight()

выходы для оценки: {'ctr': 'ctr', 'cr_general': 'cr', 'cr_adj': 'cr'}
сравнение: ['cr_adj vs cr_general']
PREFLIGHT: OK


## 5. Задача osiris

In [15]:
def build_script_args():
    a = [
        "--num_machines", f"{NUM_NODES}",
        "--num_processes", f"{NUM_PROCESS}",
        "--main_process_ip", "$(MASTER_ADDR).datalab.svc.cluster.local",
        "--main_process_port", "$(MASTER_PORT)",
        "--machine_rank", "$(RANK)",
        f"{METRICS_SCRIPT_PATH}",
        "--local-root", str(LOCAL_ROOT),
        "--config-path", str(CONFIG_PATH),
        "--mdl-name", str(MDL_NAME),
        "--chkpt-name", str(CKPT_NAME),
        "--mt-config", str(MT_CONFIG_NAME),
        "--target-columns", str(TARGET_COLUMNS),
        "--heads", str(HEADS),
        "--eval-splits", str(EVAL_SPLITS),
        "--data-format", str(DATA_FORMAT),
        "--dataset-size", str(DATASET_SIZE),
        "--batch-size", str(BATCH_SIZE),
        "--seed", str(SEED),
        "--num-workers", str(NUM_WORKERS),
        "--file-fraction", str(FILE_FRACTION),
        "--modes", str(MODES),
        "--n-boot", str(N_BOOT),
        "--boot-max-rows", str(BOOT_MAX_ROWS),
        "--use-emb-flags", str(USE_EMB_FLAGS),
        "--evk-weights-path", EVK_WEIGHTS_PATH,
        "--dlt-weights-path", DLT_WEIGHTS_PATH,
        "--result-name", str(RESULT_TAG),
    ]
    if DATASET_PATH:
        a += ["--dataset-path", DATASET_PATH]
    for flag, val in (("--val-table", VAL_TABLE), ("--tst-table", TST_TABLE), ("--oot-table", OOT_TABLE)):
        if val:
            a += [flag, val]
    if MAX_BATCHES:
        a += ["--max-batches", str(MAX_BATCHES)]
    if NO_COMPARE:
        a += ["--no-compare"]
    return a


def create_job(job_name, num_gpus):
    return osiris.create(
        name=f"sc-{job_name}",
        image="registry.ca.sbrf.ru/ci02684173/ci02697916/notebooks/python3.12/cuda12.4/d-03.000.00:d-03.000.00-gigachat",
        restart=False,
        pool=POOL,
        script=METRICS_SCRIPT_PATH,
        command=["accelerate", "launch"],
        args=build_script_args(),
        envs={
            "PYTHONPATH": "",
            "OMP_NUM_THREADS": "30",
            "NCCL_DEBUG": "INFO",
            "HADOOP_CONF_DIR": "/home/datalab/nfs/hadoop",
            "KRB5_CONFIG": "/home/datalab/nfs/krb5.conf",
            "KRB5CCNAME": "/home/datalab/nfs/krb5cc",
            "ARROW_LIBHDFS_OPTS": "-Djava.security.krb5.conf=/home/datalab/nfs/krb5.conf",
            "LIBHDFS_OPTS": "-Djava.security.krb5.conf=/home/datalab/nfs/krb5.conf",
            "HADOOP_OPTS": "-Djava.security.krb5.conf=/home/datalab/nfs/krb5.conf",
            "LD_PRELOAD": "/usr/lib64/libnss_wrapper.so",
            "NSS_WRAPPER_PASSWD": "/home/datalab/nfs/nss/passwd",
            "NSS_WRAPPER_GROUP": "/etc/group",
            "CLASSPATH": f":/home/datalab/nfs/hadoop:" + os.environ["CLASSPATH"].split(":", 2)[-1],
            "TORCH_DISTRIBUTED_DEBUG": "DETAIL",
            "NCCL_SOCKET_FAMILY": "AF_INET",
            "GLOO_SOCKET_FAMILY": "AF_INET",
            "NCCL_SOCKET_IFNAME": "eth0",
            "GLOO_SOCKET_IFNAME": "eth0",
            "NCCL_IB_DISABLE": "1",
            "NCCL_P2P_DISABLE": "1",
            "NCCL_SHM_DISABLE": "1",
        },
        num_nodes=NUM_NODES,
        num_gpus=num_gpus,
        type="pytorchjob",
    )

print(" ".join(f"'{x}'" if (" " in x or x[:1] in "{[") else x for x in build_script_args()))

--num_machines 3 --num_processes 12 --main_process_ip $(MASTER_ADDR).datalab.svc.cluster.local --main_process_port $(MASTER_PORT) --machine_rank $(RANK) '/home/datalab/nfs/Omni/SCORE OMNI/MDMH score/CR CTR EXPERTS CHNL score/mt_ddp_metrics.py' --local-root /home/datalab/nfs/Omni/ --config-path /home/datalab/nfs/Omni/configs_weights/omni_config.json --mdl-name omni_val_cr_ctr_mt_adapter --chkpt-name omni_val_cr_ctr_mt_adapter_ep1_weights.pt --mt-config omni_val_cr_ctr_mt_adapter_mt_config.json --target-columns trgt_fin,trgt_ctr --heads auto --eval-splits oot --data-format raw --dataset-size 202666520 --batch-size 128 --seed 42 --num-workers 2 --file-fraction 1.0 --modes general,product,channel,product_channel --n-boot 200 --boot-max-rows 300000 --use-emb-flags '{"romashka_emb": true,
        "evk_emb": true,
        "prod_hier_emb": true,
        "prod_desc_emb": true
    }' --evk-weights-path evk_weights.pt --dlt-weights-path dlt_weights.pt --result-name metrics --oot-table sdi_cr_ctr_

## 6. Запуск

`DRY_RUN = True` ничего не создаёт. После `DRY_RUN = False` выполните эту ячейку и следующую.

In [16]:
JOB_NAME = f"mt-metrics-{RUN_MODE}"

if DRY_RUN:
    job_name = None
    print("DRY_RUN: задача не создавалась")
else:
    HEADS_RESOLVED = preflight()
    job = create_job(JOB_NAME, NUM_GPUS)
    print(job)
    job_name = job["job"]

выходы для оценки: {'ctr': 'ctr', 'cr_general': 'cr', 'cr_adj': 'cr'}
сравнение: ['cr_adj vs cr_general']
PREFLIGHT: OK
{'job': 'training-21835579-pytorchjob-sc-mt-metrics-full', 'uuid': '96459bd6-8c8b-4083-8680-ec5ff361b2b7', 'state': 'Created', 'time': '2026-10-19T13:44:07Z', 'projects': []}


## 7. Мониторинг, разбор логов и результаты

In [ ]:
def _find_job(name):
    for j in osiris.list()["jobs"]:
        if j["job"] == name:
            return j
    return None


def check_report(text):
    issues = []
    for pat in (r"\[EVAL\] выходы:.*", r"\[CKPT\] Загружено.*", r"\[EVAL\] строк всего:.*", r"\[METRICS\] Saved:.*"):
        m = re.search(pat, text)
        print(m.group(0)[:300] if m else f"(нет строки по шаблону {pat})")
        if not m and "Saved" in pat:
            issues.append("нет строки [METRICS] Saved: результат не записан")
    for w in re.findall(r"\[WARN\][^\r\n]*|\[CKPT\] WARNING[^\r\n]*", text):
        issues.append(w[:250])
    if "Traceback" in text:
        issues.append(f"в логе есть Traceback ({text.count('Traceback')})")
    if "в чекпоинте нет весов MT-головы" in text:
        issues.append("в чекпоинте нет весов mt.* (не тот чекпоинт или конфиг)")
    return issues


def show_results(split=None, tag=None):
    split = split or EVAL_SPLITS.split(",")[0].strip()
    tag = tag or RESULT_TAG
    files = sorted(glob.glob(f"{METRICS_DIR}/mt_*_{split}_{tag}.xlsx"), key=os.path.getmtime)
    if not files:
        print("нет файлов результатов:", f"{METRICS_DIR}/mt_*_{split}_{tag}.xlsx")
        return None
    path = files[-1]
    print("файл:", path, time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(path))))
    xl = pd.read_excel(path, sheet_name=None)
    m = xl["metrics"]
    show = [c for c in ("model", "slice_name", "communications", "avg_cr", "gini", "pr_auc", "logloss", "ece") if c in m.columns]
    print("\nметрики (общие и по каналам):")
    print(m[m["slice_type"].isin(["general", "channel"])][show].round(4).to_string(index=False))
    if "compare" in xl:
        c = xl["compare"]
        print("\nсравнение a - b (95% бутстрэп-интервал):")
        print(c[["пара", "канал", "строк", "d_roc_auc", "d_roc_lo", "d_roc_hi", "вывод_roc", "d_pr_auc", "d_pr_lo", "d_pr_hi", "вывод_pr"]].round(4).to_string(index=False))
    return xl


if job_name:
    my_job = _find_job(job_name)
    while my_job is None or my_job["state"] not in ("Succeeded", "Failed"):
        my_job = _find_job(job_name)
        if my_job is not None:
            print(my_job["state"], end="\r")
        time.sleep(5)
    print("Состояние:", my_job["state"])
    try:
        out = osiris.logs(job_name, timestamps=False, is_master=True, tail_lines=20000)
        text = out if isinstance(out, str) else (str(out) if out is not None else "")
    except Exception as e:
        text = ""
        print("не удалось получить логи автоматически:", e)
    if text:
        issues = check_report(text)
        print("\nЗамечания:" if issues else "\nЗамечаний нет")
        for i in issues:
            print(" -", i)
    if my_job["state"] == "Succeeded":
        xl = show_results()
else:
    print("задача не создавалась (DRY_RUN)")

In [11]:
# показать результаты уже готового расчёта (без запуска задачи)
# xl = show_results(split="val", tag=RESULT_TAG)

In [12]:
# логи мастера вручную (ищите [EVAL], [CKPT], [WARN], Traceback)
if job_name:
    osiris.logs(job_name, timestamps=False, is_master=True, tail_lines=300)
else:
    print("задача не создавалась (DRY_RUN)")

In [29]:
# удаление задачи
if job_name: osiris.delete(job_name)